In [2]:
#access the close talk (M2) microphone for baseline whisper and the corresponding transcripts.

In [3]:
#imports
import os
import pandas as pd
import librosa

In [4]:
#paths
HDSD_ROOT = r"C:\Users\edwin\OneDrive\Desktop\Capstone\hindi indic\HDSD"
AUDIO_ROOT = os.path.join(HDSD_ROOT, "hindi_sent")
TRANSCRIPT_ROOT = os.path.join(HDSD_ROOT, "WORD")

In [5]:
dataset_rows = []

for speaker in os.listdir(AUDIO_ROOT):
    speaker_audio_dir = os.path.join(AUDIO_ROOT, speaker)
    if not os.path.isdir(speaker_audio_dir):
        continue

    for file in os.listdir(speaker_audio_dir):
        # Use only close talk microphone
        if not file.endswith("_M2.wav"):
            continue

        audio_path = os.path.join(speaker_audio_dir, file)

        # Example:
        # CF00_S1_H01_M2.wav
        # -> CF00_S1_H01
        transcript_id = file.replace("_M2.wav", "")
        transcript_path = os.path.join(
            TRANSCRIPT_ROOT,
            speaker,
            transcript_id + ".txt"
        )
        if not os.path.exists(transcript_path):
            print(f"Missing transcript: {transcript_path}")
            continue
        words = []

        with open(transcript_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                words.append(parts[2])

        transcript = " ".join(words)
        dataset_rows.append(
            {
                "audio": audio_path,
                "text": transcript,
                "speaker": speaker,
                "utterance": transcript_id,
                "mic": "M2"
            }
        )

In [6]:
df = pd.DataFrame(dataset_rows)

In [7]:
print("Total samples:", len(df))
display(df.head())

Total samples: 2006


,audio,text,speaker,utterance,mic
0,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,aapakei hindii pasanda karanei para khushii huii,CF00,CF00_S1_H01,M2
1,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,isakei badalei mein tuma kuchha aura maanaga loo,CF00,CF00_S1_H02,M2
2,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,chikitsaa kaa artha hootaa hai ilaaja,CF00,CF00_S1_H03,M2
3,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,isei aisei hii jaarii rakhein,CF00,CF00_S1_H04,M2
4,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,madada kei liyei bahuta dhanyavaada,CF00,CF00_S1_H05,M2


In [8]:
#verifyiing dataset
print(df.iloc[0]["audio"])
print(df.iloc[0]["text"])

C:\Users\edwin\OneDrive\Desktop\Capstone\hindi indic\HDSD\hindi_sent\CF00\CF00_S1_H01_M2.wav
aapakei hindii pasanda karanei para khushii huii


In [9]:
#checking speaker distribution
print(df["speaker"].nunique())
print(df["speaker"].value_counts().head())

63
speaker
M42     90
M15     67
M62     60
M55     60
CM04    30
Name: count, dtype: int64


In [10]:
#convert to huggingface dataset
from datasets import Dataset
hf_dataset = Dataset.from_pandas(df)
print(hf_dataset)
print(hf_dataset.features)

C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['audio', 'text', 'speaker', 'utterance', 'mic'],
    num_rows: 2006
})
{'audio': Value('string'), 'text': Value('string'), 'speaker': Value('string'), 'utterance': Value('string'), 'mic': Value('string')}


In [11]:
print(len(df))
print(df.head())
print(df.dtypes)

2006
                                               audio  \
0  C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...   
1  C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...   
2  C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...   
3  C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...   
4  C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...   

                                               text speaker    utterance mic  
0  aapakei hindii pasanda karanei para khushii huii    CF00  CF00_S1_H01  M2  
1  isakei badalei mein tuma kuchha aura maanaga loo    CF00  CF00_S1_H02  M2  
2             chikitsaa kaa artha hootaa hai ilaaja    CF00  CF00_S1_H03  M2  
3                     isei aisei hii jaarii rakhein    CF00  CF00_S1_H04  M2  
4               madada kei liyei bahuta dhanyavaada    CF00  CF00_S1_H05  M2  
audio        object
text         object
speaker      object
utterance    object
mic          object
dtype: object


In [12]:
print(hf_dataset[0]["audio"])
print(hf_dataset[0]["text"])

C:\Users\edwin\OneDrive\Desktop\Capstone\hindi indic\HDSD\hindi_sent\CF00\CF00_S1_H01_M2.wav
aapakei hindii pasanda karanei para khushii huii


In [13]:
print(hf_dataset.format)
print(hf_dataset[0])

{'type': None, 'format_kwargs': {}, 'columns': ['audio', 'text', 'speaker', 'utterance', 'mic'], 'output_all_columns': False}
{'audio': 'C:\\Users\\edwin\\OneDrive\\Desktop\\Capstone\\hindi indic\\HDSD\\hindi_sent\\CF00\\CF00_S1_H01_M2.wav', 'text': 'aapakei hindii pasanda karanei para khushii huii', 'speaker': 'CF00', 'utterance': 'CF00_S1_H01', 'mic': 'M2'}


In [14]:
#sanity check
print("Total samples:", len(df))
print("Speakers:", df["speaker"].nunique())
print(df["mic"].value_counts())
df.groupby("speaker").size().describe()

Total samples: 2006
Speakers: 63
mic
M2    2006
Name: count, dtype: int64


count    63.00000
mean     31.84127
std      10.84228
min      10.00000
25%      30.00000
50%      30.00000
75%      30.00000
max      90.00000
dtype: float64

In [15]:
#check transcript lengths
df["n_words"] = df["text"].str.split().str.len()
print(df["n_words"].describe())

count    2006.000000
mean        6.399302
std         1.941945
min         0.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        19.000000
Name: n_words, dtype: float64


In [16]:
#remove the rows with no utterances
empty_rows = df[df["n_words"] == 0]
print(len(empty_rows))
display(empty_rows.head())

4


,audio,text,speaker,utterance,mic,n_words
1965,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,,M62,M62_S1_H20,M2,0
1966,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,,M62,M62_S1_H21,M2,0
1971,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,,M62,M62_S1_H26,M2,0
1974,C:\Users\edwin\OneDrive\Desktop\Capstone\hindi...,,M62,M62_S1_H29,M2,0


In [17]:
#drop these
df = df[df["n_words"] > 0].reset_index(drop=True)

In [18]:
#check again
df["n_words"] = df["text"].str.split().str.len()
print(df["n_words"].describe())

count    2002.000000
mean        6.412088
std         1.922671
min         2.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        19.000000
Name: n_words, dtype: float64


### hinglish to devnagari conversion

In [31]:

!pip install indic-transliteration

  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)

   ------------------------------ --------- 3/4 [indic-transliteration]
   ------------------------------ --------- 3/4 [indic-transliteration]
   ------------------------------ --------- 3/4 [indic-transliteration]
   ------------------------------ --------- 3/4 [indic-transliteration]
   ---------------------------------------- 4/4 [indic-transliteration]



In [33]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def hinglish_to_devanagari(text):
    try:
        return transliterate(
            text,
            sanscript.ITRANS,
            sanscript.DEVANAGARI
        )
    except:
        return text

df["text_devnagari"] = df["text"].apply(hinglish_to_devanagari)

In [35]:
for i in range(10):
    print("HINGLISH :", df.iloc[i]["text"])
    print("DEVANAGARI:", df.iloc[i]["text_devnagari"])
    print()

HINGLISH : aapakei hindii pasanda karanei para khushii huii
DEVANAGARI: आपकेइ हिन्दी पसन्द करनेइ पर खुशी हुई

HINGLISH : isakei badalei mein tuma kuchha aura maanaga loo
DEVANAGARI: इसकेइ बदलेइ मेइन् तुम कुछ और मानग लू

HINGLISH : chikitsaa kaa artha hootaa hai ilaaja
DEVANAGARI: चिकित्सा का अर्थ हूता है इलाज

HINGLISH : isei aisei hii jaarii rakhein
DEVANAGARI: इसेइ ऐसेइ ही जारी रखेइन्

HINGLISH : madada kei liyei bahuta dhanyavaada
DEVANAGARI: मदद केइ लियेइ बहुत धन्यवाद

HINGLISH : aba joo main kahataa huuna usakaa paalana karoo
DEVANAGARI: अब जू मैन् कहता हून उसका पालन करू

HINGLISH : bhaarata mein aisaa nahiin huaa hai
DEVANAGARI: भारत मेइन् ऐसा नहीन् हुआ है

HINGLISH : isa sahayooga heitu bhii bahuta dhanyavaada
DEVANAGARI: इस सहयूग हेइतु भी बहुत धन्यवाद

HINGLISH : para yaha sanbhava naa hoo sakaa
DEVANAGARI: पर यह सन्भव ना हू सका

HINGLISH : meirii baatoon ki oora dhyaana diijiyei
DEVANAGARI: मेइरी बातून् कि ऊर ध्यान दीजियेइ



In [36]:
df.to_csv(
    "hdsd_devnagari.csv",
    index=False,
    encoding="utf-8-sig"
)

In [38]:
for i in range(20):
    print("HINGLISH :", df.iloc[i]["text"])
    print("DEVANAGARI:", df.iloc[i]["text_devnagari"])
    print("-"*60)

HINGLISH : aapakei hindii pasanda karanei para khushii huii
DEVANAGARI: आपकेइ हिन्दी पसन्द करनेइ पर खुशी हुई
------------------------------------------------------------
HINGLISH : isakei badalei mein tuma kuchha aura maanaga loo
DEVANAGARI: इसकेइ बदलेइ मेइन् तुम कुछ और मानग लू
------------------------------------------------------------
HINGLISH : chikitsaa kaa artha hootaa hai ilaaja
DEVANAGARI: चिकित्सा का अर्थ हूता है इलाज
------------------------------------------------------------
HINGLISH : isei aisei hii jaarii rakhein
DEVANAGARI: इसेइ ऐसेइ ही जारी रखेइन्
------------------------------------------------------------
HINGLISH : madada kei liyei bahuta dhanyavaada
DEVANAGARI: मदद केइ लियेइ बहुत धन्यवाद
------------------------------------------------------------
HINGLISH : aba joo main kahataa huuna usakaa paalana karoo
DEVANAGARI: अब जू मैन् कहता हून उसका पालन करू
------------------------------------------------------------
HINGLISH : bhaarata mein aisaa nahiin huaa hai
DEVANAGAR

In [39]:
df["n_chars_dev"] = df["text_devnagari"].str.len()
print(df["n_chars_dev"].describe())

count    2002.000000
mean       29.420579
std         9.789026
min         5.000000
25%        24.000000
50%        27.000000
75%        34.000000
max        88.000000
Name: n_chars_dev, dtype: float64


In [42]:
#we need to split the data in such a way so as to not allow the model to memorise speaking patterns of some speakers which will artificially decrease the WER/CER of that speaker
#this will skew the results and not provide a good estimate of the fine tuned model's capabilities.
#Therefore split the data into train-test-val speaker wise.

from sklearn.model_selection import train_test_split

speakers = sorted(df["speaker"].unique())

train_speakers, temp_speakers = train_test_split(
    speakers,
    test_size=0.20,
    random_state=42
)

val_speakers, test_speakers = train_test_split(
    temp_speakers,
    test_size=0.50,
    random_state=42
)

train_df = df[df["speaker"].isin(train_speakers)].reset_index(drop=True)
val_df   = df[df["speaker"].isin(val_speakers)].reset_index(drop=True)
test_df  = df[df["speaker"].isin(test_speakers)].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))

1633 159 210


In [43]:
#verify no leakage
print(set(train_df["speaker"]) & set(val_df["speaker"]))
print(set(train_df["speaker"]) & set(test_df["speaker"]))
print(set(val_df["speaker"]) & set(test_df["speaker"]))

set()
set()
set()


In [44]:
print("Train samples:", len(train_df))
print("Val samples:", len(val_df))
print("Test samples:", len(test_df))

Train samples: 1633
Val samples: 159
Test samples: 210


In [45]:
#convert to hf datasets
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['audio', 'text', 'speaker', 'utterance', 'mic', 'n_words', 'text_devnagari', 'n_chars_dev'],
    num_rows: 1633
})
Dataset({
    features: ['audio', 'text', 'speaker', 'utterance', 'mic', 'n_words', 'text_devnagari', 'n_chars_dev'],
    num_rows: 159
})
Dataset({
    features: ['audio', 'text', 'speaker', 'utterance', 'mic', 'n_words', 'text_devnagari', 'n_chars_dev'],
    num_rows: 210
})


In [46]:
for i in range(5):
    print(test_ds[i]["text_devnagari"])

आपकेइ हिन्दी पसन्द करनेइ पर खुशी हुई
इसकेइ बदलेइ मेइन् तुम कुछ और मानग लू
चिकित्सा का अर्थ हूता है इलाज
इसेइ ऐसेइ ही जारी रखेइन्
मदद केइ लियेइ बहुत धन्यवाद


In [47]:
train_df.to_csv("hdsd_train.csv", index=False)
val_df.to_csv("hdsd_val.csv", index=False)
test_df.to_csv("hdsd_test.csv", index=False)

In [48]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="hi",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8318.28it/s]


In [49]:
#test one example (avoidable)
import librosa

audio, sr = librosa.load(
    test_ds[0]["audio"],
    sr=16000
)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(device)

with torch.no_grad():
    predicted_ids = model.generate(input_features)

prediction = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("REF :", test_ds[0]["text_devnagari"])
print("PRED:", prediction)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


REF : आपकेइ हिन्दी पसन्द करनेइ पर खुशी हुई
PRED:  आपके हिंदी पसन्द कने पर खुषी हुई


In [50]:
for i in range(20):
    audio, sr = librosa.load(test_ds[i]["audio"], sr=16000)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        pred_ids = model.generate(
            inputs.input_features.to(device)
        )

    pred = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )[0]

    print("=" * 80)
    print("REF :", test_ds[i]["text_devnagari"])
    print("PRED:", pred)

REF : आपकेइ हिन्दी पसन्द करनेइ पर खुशी हुई
PRED:  आपके हिंदी पसन्द कने पर खुषी हुई
REF : इसकेइ बदलेइ मेइन् तुम कुछ और मानग लू
PRED:  इसके बदले में तुम कुछ और मांगलो.
REF : चिकित्सा का अर्थ हूता है इलाज
PRED:  चिकिट्चा का अर्थ होता है, इलाज
REF : इसेइ ऐसेइ ही जारी रखेइन्
PRED:  इसे आँईसे ही जारी रकें
REF : मदद केइ लियेइ बहुत धन्यवाद
PRED:  मदद के लिए बहुत द्धन्निवाद
REF : अब जू मैन् कहता हून उसका पालन करू
PRED:  अप जो में कहता हूँ उसका पालन करो
REF : भारत मेइन् ऐसा नहीन् हुआ है
PRED:  बारत्द में एसा नहीं हुए है
REF : इस सहयूग हेइतु भी बहुत धन्यवाद
PRED:  इसे सहयोग हे तुब ही बहुत द्धन्नवाद
REF : पर यह सन्भव ना हू सका
PRED:  पर ये समबव नहो सका
REF : मेइरी बातून् कि ऊर ध्यान दीजियेइ
PRED:  मेरी बातों की और दहन दीजीए
REF : यहान् सेइ सन्घर्ष का दौर शुरू हुआ
PRED:  यहाँ से संगर्ष्का दोर शुरू हूँ आँँ
REF : दूनून् मेइन् बदा अन्तर था
PRED:  दोनो में बड़ा अंदर ता
REF : आप सभी केइ सहयूग केइ लियेइ बहुत धन्यवाद
PRED:  अप सभी के सहयों के ले बहुत द्धन्निवाद
REF : बहुत सेइ लूग ऐसा कर रहेइ है
PRED:  बहु


PROBLEM: the transcript is noisy. As a result the transliteration is producing outputs which are not in Hindi format such as : आपकेइ, करनेइ, मेइन् instead of आपके, करने, में
SOLUTION: so we need to build a normalisation layer on the original df["text"] before we transliterate from Romanized hindi(hinglish) to Devnagari Hindi. Currently the Whisper model is performing surprisingly well on the HDSD test data.

Lets create the normalization layer.
First, find the patterns in the dataset

In [51]:
from collections import Counter
import re

patterns = Counter()

for txt in df["text"]:
    words = txt.split()
    for word in words:
        for pat in re.findall(r"(ei|ii|oo|uu|ai|au|oon|ein)", word):
            patterns[pat] += 1

print(patterns.most_common())

[('ei', 2926), ('ii', 1653), ('oo', 1336), ('ai', 938), ('uu', 463), ('au', 134)]


Inspect actual words containing these patterns

In [52]:
from collections import Counter

patterns = ["ei", "ii", "oo", "uu", "ai", "au"]

for pat in patterns:
    words = Counter()

    for txt in df["text"]:
        for word in txt.split():
            if pat in word:
                words[word] += 1

    print(f"\n=== {pat} ===")
    for w, c in words.most_common(30):
        print(f"{w:20} {c}")


=== ei ===
kei                  328
mein                 327
sei                  264
liyei                194
rakhein              132
isei                 130
aisei                72
meirii               72
aagei                70
heitu                69
karanei              68
badalei              68
diijiyei             68
usasei               68
krikeita             68
jaanei               68
kitanei              68
isakei               67
jaaei                67
kheila               67
aapanei              67
deikhaa              67
padhtei              67
rahei                66
aapakei              65
eika                 65
sandeisha            65
kiijiei              65
unhoonnei            64

=== ii ===
nahiin               275
hii                  249
jaarii               133
bhii                 132
jaldii               132
meirii               72
hindii               68
diijiyei             68
sabhii               68
khushii              67
abhii                67
huii 

The data shows HDSD is using a phonetic Romanization scheme, not random spelling.

Safe Normalizations

Rule 1:
Most ii endings should become i

Rule 2:
Most ei endings should become e

"| normalizer_v1 |"


In [53]:
import re

def normalize_hdsd(text):

    text = text.lower()

    # End-of-word fixes
    text = re.sub(r"ei\b", "e", text)
    text = re.sub(r"ii\b", "i", text)

    return text

In [55]:
df["text_norm"] = df["text"].apply(normalize_hdsd)

df["text_devnagari"] = df["text_norm"].apply(
    lambda x: transliterate(
        x,
        sanscript.ITRANS,
        sanscript.DEVANAGARI
    )
)

In [60]:
for i in range(20):
    print("HINGLISH :", df.iloc[i]["text"])
    print("NORMAL   :", df.iloc[i]["text_norm"])
    print("DEV      :", df.iloc[i]["text_devnagari"])
    print("-" * 80)

HINGLISH : aapakei hindii pasanda karanei para khushii huii
NORMAL   : aapake hindi pasanda karane para khushi hui
DEV      : आपके हिन्दि पसन्द करने पर खुशि हुइ
--------------------------------------------------------------------------------
HINGLISH : isakei badalei mein tuma kuchha aura maanaga loo
NORMAL   : isake badale mein tuma kuchha aura maanaga loo
DEV      : इसके बदले मेइन् तुम कुछ और मानग लू
--------------------------------------------------------------------------------
HINGLISH : chikitsaa kaa artha hootaa hai ilaaja
NORMAL   : chikitsaa kaa artha hootaa hai ilaaja
DEV      : चिकित्सा का अर्थ हूता है इलाज
--------------------------------------------------------------------------------
HINGLISH : isei aisei hii jaarii rakhein
NORMAL   : ise aise hi jaari rakhein
DEV      : इसे ऐसे हि जारि रखेइन्
--------------------------------------------------------------------------------
HINGLISH : madada kei liyei bahuta dhanyavaada
NORMAL   : madada ke liye bahuta dhanyavaada
DEV     

In [61]:
df.to_csv(
    "hdsd_devnagari.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved dataset")

Saved dataset


rebuilding the speaker splits same as before

In [62]:
train_df = df[df["speaker"].isin(train_speakers)].reset_index(drop=True)
val_df   = df[df["speaker"].isin(val_speakers)].reset_index(drop=True)
test_df  = df[df["speaker"].isin(test_speakers)].reset_index(drop=True)

print(len(train_df))
print(len(val_df))
print(len(test_df))

1633
159
210


In [63]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

In [64]:
for i in range(5):
    print(test_ds[i]["text_devnagari"])

आपके हिन्दि पसन्द करने पर खुशि हुइ
इसके बदले मेइन् तुम कुछ और मानग लू
चिकित्सा का अर्थ हूता है इलाज
इसे ऐसे हि जारि रखेइन्
मदद के लिये बहुत धन्यवाद


In [65]:
print(train_ds.column_names)

['audio', 'text', 'speaker', 'utterance', 'mic', 'n_words', 'text_devnagari', 'n_chars_dev', 'text_norm']


In [66]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="hi",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(device)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 7635.48it/s]


cuda


In [68]:
for i in range(10):

    sample = test_ds[i]

    audio, sr = librosa.load(
        sample["audio"],
        sr=16000
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        pred_ids = model.generate(
            inputs.input_features.to(device)
        )

    pred = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )[0]

    print("=" * 80)
    print("REF :", sample["text_devnagari"])
    print("PRED:", pred)

REF : आपके हिन्दि पसन्द करने पर खुशि हुइ
PRED:  आपके हिंदी पसन्द कने पर खुषी हुई
REF : इसके बदले मेइन् तुम कुछ और मानग लू
PRED:  इसके बदले में तुम कुछ और मांगलो.
REF : चिकित्सा का अर्थ हूता है इलाज
PRED:  चिकिट्चा का अर्थ होता है, इलाज
REF : इसे ऐसे हि जारि रखेइन्
PRED:  इसे आँईसे ही जारी रकें
REF : मदद के लिये बहुत धन्यवाद
PRED:  मदद के लिए बहुत द्धन्निवाद
REF : अब जू मैन् कहता हून उसका पालन करू
PRED:  अप जो में कहता हूँ उसका पालन करो
REF : भारत मेइन् ऐसा नहीन् हुआ है
PRED:  बारत्द में एसा नहीं हुए है
REF : इस सहयूग हेइतु भि बहुत धन्यवाद
PRED:  इसे सहयोग हे तुब ही बहुत द्धन्नवाद
REF : पर यह सन्भव ना हू सका
PRED:  पर ये समबव नहो सका
REF : मेइरि बातून् कि ऊर ध्यान दीजिये
PRED:  मेरी बातों की और दहन दीजीए


Now run inference on the full test set.

In [69]:
import librosa

predictions = []
references = []

for idx, sample in enumerate(test_ds):

    audio, sr = librosa.load(
        sample["audio"],
        sr=16000
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        pred_ids = model.generate(input_features)

    pred = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(pred.strip())
    references.append(sample["text_devnagari"].strip())

    if (idx + 1) % 25 == 0:
        print(f"Processed {idx+1}/{len(test_ds)}")

Processed 25/210
Processed 50/210
Processed 75/210
Processed 100/210
Processed 125/210
Processed 150/210
Processed 175/210
Processed 200/210


In [71]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

wer_baseline = wer_metric.compute(
    predictions=predictions,
    references=references
)

cer_baseline = cer_metric.compute(
    predictions=predictions,
    references=references
)

print("Baseline Whisper-small")
print("WER:", round(wer_baseline, 4))
print("CER:", round(cer_baseline, 4))

Baseline Whisper-small
WER: 1.3996
CER: 1.0809


save results

In [74]:
baseline_results = {
    "dataset": "HDSD",
    "model": "openai/whisper-small",
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "test_samples": len(test_ds),
    "wer_baseline": float(wer_baseline),
    "cer_baseline": float(cer_baseline),
}

import json

with open("hdsd_baseline_results_rawTransliterated.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2, ensure_ascii=False)

In [73]:
for i in range(20):
    print(len(references[i].split()))
    print(len(predictions[i].split()))
    print("REF :", references[i])
    print("PRED:", predictions[i])
    print()

7
7
REF : आपके हिन्दि पसन्द करने पर खुशि हुइ
PRED: आपके हिंदी पसन्द कने पर खुषी हुई

8
7
REF : इसके बदले मेइन् तुम कुछ और मानग लू
PRED: इसके बदले में तुम कुछ और मांगलो.

6
6
REF : चिकित्सा का अर्थ हूता है इलाज
PRED: चिकिट्चा का अर्थ होता है, इलाज

5
5
REF : इसे ऐसे हि जारि रखेइन्
PRED: इसे आँईसे ही जारी रकें

5
5
REF : मदद के लिये बहुत धन्यवाद
PRED: मदद के लिए बहुत द्धन्निवाद

8
8
REF : अब जू मैन् कहता हून उसका पालन करू
PRED: अप जो में कहता हूँ उसका पालन करो

6
6
REF : भारत मेइन् ऐसा नहीन् हुआ है
PRED: बारत्द में एसा नहीं हुए है

6
7
REF : इस सहयूग हेइतु भि बहुत धन्यवाद
PRED: इसे सहयोग हे तुब ही बहुत द्धन्नवाद

6
5
REF : पर यह सन्भव ना हू सका
PRED: पर ये समबव नहो सका

6
6
REF : मेइरि बातून् कि ऊर ध्यान दीजिये
PRED: मेरी बातों की और दहन दीजीए

7
7
REF : यहान् से सन्घर्ष का दौर शुरू हुआ
PRED: यहाँ से संगर्ष्का दोर शुरू हूँ आँँ

5
5
REF : दूनून् मेइन् बदा अन्तर था
PRED: दोनो में बड़ा अंदर ता

8
8
REF : आप सभि के सहयूग के लिये बहुत धन्यवाद
PRED: अप सभी के सहयों के ले बहुत द्धन्निवाद

7
7
R